| Order_ID | Customer_ID | Sales_Amount   | Order_Date |
| -------- | ----------- | -------------- | ---------- |
| O101     | C001        | 4500           | 12-01-2024 |
| O102     | C002        | NULL           | 15-01-2024 |
| O103     | C003        | 3200           | 2024/01/18 |
| O101     | C001        | 4500           | 12-01-2024 |
| O104     | C004        | Three Thousand | 20-01-2024 |
| O105     | C005        | 5100           | 25-01-2024 |


Q1. Data Understanding

Identify all data quality issues present in the dataset that can cause problems during data loading.

| Data Quality Issue               | Affected Record(s) | Description                                                                 | Impact on Data Loading                                                    |
| -------------------------------- | ------------------ | --------------------------------------------------------------------------- | ------------------------------------------------------------------------- |
| **Duplicate Record**             | Order_ID **O101**  | The same order appears twice.                                               | Causes duplicate entries and inaccurate reporting.                        |
| **Missing Value (NULL)**         | Order_ID **O102**  | `Sales_Amount` is NULL.                                                     | May cause errors in calculations and aggregations.                        |
| **Inconsistent Date Format**     | Order_ID **O103**  | Date is `2024/01/18` while others use `DD-MM-YYYY`.                         | Can lead to date parsing or conversion errors.                            |
| **Invalid Data Type**            | Order_ID **O104**  | `Sales_Amount` contains text (`Three Thousand`) instead of a numeric value. | Prevents numeric calculations and database insertion into numeric fields. |
| **Mixed Data Types in a Column** | `Sales_Amount`     | Column contains both numbers and text values.                               | Violates data type consistency and may cause ETL failures.                |


Q2. Primary Key Validation

Assume Order_ID is the Primary Key.

a) Is the dataset violating the Primary Key rule?

b) Which record(s) cause this violation?

### Answer 2 Primary Key Validation

**Assumption:** `Order_ID` is the **Primary Key**.

#### a) Is the dataset violating the Primary Key rule?

**Yes.** The dataset violates the Primary Key rule because the value **O101** appears more than once. A Primary Key must contain **unique** and **non-NULL** values for every record.

#### b) Which record(s) cause this violation?

The duplicate **Order_ID = O101** causes the violation.

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| -------- | ----------- | ------------ | ---------- |
| O101     | C001        | 4500         | 12-01-2024 |
| O101     | C001        | 4500         | 12-01-2024 |

### Conclusion

* **Primary Key Violated:** **Yes**
* **Reason:** Duplicate `Order_ID`
* **Violating Record(s):** Both records with **Order_ID = O101**


Q3. Missing Value Analysis

Which column(s) contain missing values?

a) List the affected records

b) Explain why loading these records without handling missing values is risky

### Answer 3. Missing Value Analysis

The dataset contains a missing value in the **`Sales_Amount`** column.

### a) Affected Record(s)

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| -------- | ----------- | ------------ | ---------- |
| O102     | C002        | **NULL**     | 15-01-2024 |

### b) Why is loading these records without handling missing values risky?

Loading records with missing values can cause several problems:

* **Incorrect calculations:** Total sales, average sales, and other reports may produce inaccurate results.
* **ETL failures:** If the target database defines `Sales_Amount` as **NOT NULL**, the record will fail to load.
* **Poor data quality:** Missing values reduce the reliability and completeness of the dataset.
* **Business decision errors:** Reports and dashboards based on incomplete data can lead to incorrect business decisions.
* **Issues in data analysis:** Missing values may affect forecasting, trend analysis, and machine learning models.

### Summary

* **Column with Missing Value:** `Sales_Amount`
* **Affected Record:** `Order_ID = O102`
* **Recommended Action:** Handle the missing value before loading the data (e.g., replace with an appropriate value, estimate/impute it, or remove the record if justified).


Q4. Data Type Validation

Identify records where Sales_Amount violates expected data type rules.

a) Which record(s) will fail numeric validation?

b) What would happen if this dataset is loaded into a SQL table with Sales_amount as DECIMAL?

###Answer 4  Data Type Validation

**Expected Data Type:** `Sales_Amount` should be **DECIMAL (Numeric)**.

### a) Which record(s) will fail numeric validation?

The following record violates the numeric data type rule:

| Order_ID | Customer_ID | Sales_Amount       | Reason                                    |
| -------- | ----------- | ------------------ | ----------------------------------------- |
| **O104** | C004        | **Three Thousand** | Contains text instead of a numeric value. |

> **Note:** The record **O102** contains **NULL**. It is a missing value, not a data type error. It will only cause an issue if the `Sales_Amount` column is defined as **NOT NULL**.

### b) What would happen if this dataset is loaded into a SQL table with `Sales_Amount` as `DECIMAL`?

If `Sales_Amount` is defined as **DECIMAL**:

* The record with **Order_ID O104** will **fail** because `"Three Thousand"` cannot be converted to a numeric value.
* The ETL process may reject that row or stop with a **data type conversion error**, depending on the database and ETL configuration.
* The **NULL** value in **O102** will load successfully **only if** the `DECIMAL` column allows `NULL`; otherwise, it will also fail due to the **NOT NULL** constraint.

### Summary

| Record   | Issue                                            | Result                                                                                       |
| -------- | ------------------------------------------------ | -------------------------------------------------------------------------------------------- |
| **O104** | Text value (`Three Thousand`) in a numeric field | Fails numeric validation and causes a conversion error.                                      |
| **O102** | NULL value                                       | Loads only if the `DECIMAL` column allows `NULL`; otherwise, it fails due to the constraint. |


Q5. Date Format Consistency

The order__data  column has multiple formats.

a) List all date formats present in the dataset
b) Why is this a problem during data loading?

### Answer 5 Date Format Consistency

The **`Order_Date`** column contains multiple date formats.

### a) List all date formats present in the dataset

| Date Format    | Example                                        | Affected Record(s)     |
| -------------- | ---------------------------------------------- | ---------------------- |
| **DD-MM-YYYY** | 12-01-2024, 15-01-2024, 20-01-2024, 25-01-2024 | O101, O102, O104, O105 |
| **YYYY/MM/DD** | 2024/01/18                                     | O103                   |

### b) Why is this a problem during data loading?

Using multiple date formats can cause several issues during ETL and database loading:

* **Date parsing errors:** The ETL tool or database may not recognize all date formats correctly.
* **Incorrect date values:** A date may be interpreted incorrectly if the expected format differs from the actual format.
* **Load failures:** Records with invalid or unexpected date formats may be rejected during import.
* **Inconsistent reporting:** Sorting, filtering, and date-based analysis may produce incorrect results.
* **Data quality issues:** Mixed formats reduce consistency and make maintenance more difficult.

### Recommendation

Standardize all dates to a single format before loading, such as:

* **DD-MM-YYYY** (e.g., `18-01-2024`), or
* **YYYY-MM-DD** (ISO 8601, recommended for SQL databases), e.g., `2024-01-18`.

This ensures successful ETL processing and consistent date handling.


Q6. Load Readiness Decision

Based on the dataset condition:

a) Should this dataset be loaded directly into the database? (Yes/No)

b) Justify your answer with at least three reasons

### Answer 6 Load Readiness Decision

#### a) Should this dataset be loaded directly into the database?

**Answer:** **No.**

The dataset should **not** be loaded directly into the database because it contains multiple data quality issues that must be resolved first.

#### b) Justification (at least three reasons)

1. **Duplicate Primary Key**

   * `Order_ID` **O101** appears twice.
   * This violates the primary key constraint and can cause the database load to fail.

2. **Missing Value**

   * `Sales_Amount` for **Order_ID O102** is **NULL**.
   * If the target column is defined as **NOT NULL**, the record will fail to load. Even if NULL is allowed, it can lead to inaccurate calculations.

3. **Invalid Data Type**

   * `Sales_Amount` for **Order_ID O104** contains the text **"Three Thousand"** instead of a numeric value.
   * This will fail if the database column is of type **DECIMAL** or another numeric type.

4. **Inconsistent Date Format**

   * `Order_Date` contains both **DD-MM-YYYY** and **YYYY/MM/DD** formats.
   * This may cause date conversion errors or inconsistent data interpretation during loading.

### Conclusion

**Load Decision:** **No**

The dataset should first be cleaned by:

* Removing duplicate records.
* Handling missing values.
* Converting text values in `Sales_Amount` to numeric values.
* Standardizing all dates to a single format.

After these data quality issues are resolved, the dataset will be ready for successful loading into the database.


Q7. Pre-Load Validation Checklist

List the exact pre-load validation checks you would perform on this dataset before loading.

Answer 7
### Pre-Load Validation Checklist

Before loading the dataset into the database, the following pre-load validation checks should be performed:

| Validation Check                  | Purpose                                                                                       |
| --------------------------------- | --------------------------------------------------------------------------------------------- |
| **1. Primary Key Validation**     | Ensure `Order_ID` is unique and contains no duplicate or NULL values.                         |
| **2. Missing Value Check**        | Identify and handle missing values, especially in `Sales_Amount`.                             |
| **3. Data Type Validation**       | Verify that `Sales_Amount` contains only numeric values and `Order_Date` is a valid date.     |
| **4. Date Format Validation**     | Standardize all dates to a single format (e.g., `YYYY-MM-DD`).                                |
| **5. Duplicate Record Check**     | Detect and remove duplicate records (e.g., duplicate `O101`).                                 |
| **6. NULL Constraint Validation** | Ensure columns that are mandatory do not contain NULL values.                                 |
| **7. Range Validation**           | Check that `Sales_Amount` is greater than zero and within an acceptable business range.       |
| **8. Customer ID Validation**     | Verify that `Customer_ID` is not NULL and follows the expected format (e.g., `C001`, `C002`). |
| **9. Data Consistency Check**     | Ensure values are consistent across records and no mixed data formats exist.                  |
| **10. Schema Validation**         | Confirm that column names, order, and data types match the target database schema.            |

### Final Pre-Load Checklist

* ✅ Check for duplicate `Order_ID`s.
* ✅ Check for NULL or missing values.
* ✅ Validate numeric data in `Sales_Amount`.
* ✅ Standardize `Order_Date` format.
* ✅ Remove duplicate records.
* ✅ Validate mandatory fields.
* ✅ Verify valid `Customer_ID` format.
* ✅ Check value ranges and business rules.
* ✅ Ensure data consistency.
* ✅ Confirm compatibility with the target database schema.

Performing these validations helps ensure a successful ETL process and improves data quality before loading into the database.


Q8. Cleaning Strategy

Describe the step-by-step cleaning actions required to make this dataset load-ready.

###Answer 8 Cleaning Strategy

To make the dataset **load-ready**, perform the following data cleaning steps in order:

| Step  | Cleaning Action                               | Affected Record(s)          | Result                                                                                                    |
| ----- | --------------------------------------------- | --------------------------- | --------------------------------------------------------------------------------------------------------- |
| **1** | Remove duplicate records based on `Order_ID`. | **O101** (duplicate)        | Ensures the primary key is unique.                                                                        |
| **2** | Handle missing values in `Sales_Amount`.      | **O102**                    | Replace with an appropriate value (e.g., mean/median/business value) or remove the record if required.    |
| **3** | Correct invalid data types.                   | **O104** (`Three Thousand`) | Convert the text value to **3000** (numeric).                                                             |
| **4** | Standardize date formats.                     | **O103** (`2024/01/18`)     | Convert to a consistent format, e.g., `18-01-2024` or `2024-01-18`.                                       |
| **5** | Validate data types.                          | All records                 | Ensure `Sales_Amount` is numeric and `Order_Date` is a valid date.                                        |
| **6** | Validate primary key.                         | All records                 | Confirm every `Order_ID` is unique and not NULL.                                                          |
| **7** | Validate business rules.                      | All records                 | Ensure `Sales_Amount` is positive and `Customer_ID` follows the required format (e.g., `C001`).           |
| **8** | Perform final quality check.                  | Entire dataset              | Verify there are no duplicates, missing values, invalid data types, or inconsistent dates before loading. |

### Final Load-Ready Dataset

| Order_ID | Customer_ID |                   Sales_Amount | Order_Date |
| -------- | ----------- | -----------------------------: | ---------- |
| O101     | C001        |                           4500 | 12-01-2024 |
| O102     | C002        | 4500 *(example imputed value)* | 15-01-2024 |
| O103     | C003        |                           3200 | 18-01-2024 |
| O104     | C004        |                           3000 | 20-01-2024 |
| O105     | C005        |                           5100 | 25-01-2024 |

> **Note:** The value used for the missing `Sales_Amount` in **O102** is only an example. In practice, the replacement should follow the organization's business rules (e.g., mean, median, previous value, or another approved method).

### Summary

The dataset becomes load-ready by:

1. Removing duplicate records.
2. Handling missing values.
3. Correcting invalid data types.
4. Standardizing date formats.
5. Validating data types and business rules.
6. Performing a final quality assurance check before loading into the database.


Q9. Loading Strategy Selection

Assume this dataset represents daily sales data.

a) Should a Full Load or Incremental Load be used?
b) Justify your choice.

Answer 9
 Loading Strategy Selection

#### a) Should a Full Load or Incremental Load be used?

**Answer:** **Incremental Load**

#### b) Justify your choice

Since the dataset represents **daily sales data**, an **Incremental Load** is the preferred approach for the following reasons:

1. **Efficient Processing**

   * Only new or modified sales records are loaded each day.
   * This reduces processing time compared to reloading the entire dataset.

2. **Better Performance**

   * Incremental loading uses fewer system resources (CPU, memory, and storage).
   * It is faster and more suitable for large datasets.

3. **Reduces Duplicate Data**

   * By loading only new records (based on `Order_ID` or `Order_Date`), the risk of duplicate entries is minimized.

4. **Supports Daily ETL Operations**

   * Daily sales systems continuously generate new records.
   * Incremental loading is designed for such regular updates.

5. **Minimizes Downtime**

   * Since only a small amount of data is processed, the database remains available and ETL jobs complete more quickly.

### When to Use a Full Load

A **Full Load** should be used only when:

* Loading the data for the first time.
* Rebuilding the target table after data corruption or major structural changes.
* Performing a complete data refresh.

### Conclusion

* **Recommended Strategy:** ✅ **Incremental Load**
* **Reason:** The dataset contains **daily sales transactions**, so loading only the new or changed records is faster, more efficient, and avoids reprocessing existing data.


Q10. BI Impact Scenario

Assume this dataset was loaded without cleaning and connected to a BI dashboard.

a) What incorrect results might appear in Total Sales KPI?
b) Which records specifically would cause misleading insights?
c) Why would BI tools not detect these issues automatically?

Answer 10
### . BI Impact Scenario

Assume the dataset is loaded **without any cleaning** and connected to a BI dashboard.

### a) What incorrect results might appear in the **Total Sales KPI**?

The **Total Sales KPI** would likely be inaccurate because:

* **Duplicate records** would **inflate total sales**.
* **NULL values** would cause missing sales amounts, reducing the total or affecting calculations.
* **Text values** in a numeric column could be ignored, treated as errors, or cause calculation failures.
* As a result, the dashboard may show an **incorrect total sales value** or display errors.

**Example:**

If duplicates are counted and the text value is ignored:

* O101 = **4500**
* O102 = **NULL**
* O103 = **3200**
* O101 (Duplicate) = **4500**
* O104 = **"Three Thousand"** (not numeric)
* O105 = **5100**

Displayed Total Sales = **4500 + 3200 + 4500 + 5100 = 17,300**

However, after cleaning the data:

* O101 = **4500**
* O102 = *(handled appropriately)*
* O103 = **3200**
* O104 = **3000**
* O105 = **5100**

Correct Total Sales = **15,800** *(excluding O102 until it is properly handled)*

---

### b) Which records specifically would cause misleading insights?

| Order_ID | Issue                         | Impact on BI Dashboard                                                  |
| -------- | ----------------------------- | ----------------------------------------------------------------------- |
| **O101** | Duplicate record              | Double-counts sales, inflating Total Sales.                             |
| **O102** | NULL `Sales_Amount`           | Sales amount is missing, causing under-reporting or calculation issues. |
| **O104** | Text value (`Three Thousand`) | Cannot be summed as a number; may be ignored or produce errors.         |
| **O103** | Different date format         | May appear under the wrong date or fail date-based filtering/grouping.  |

---

### c) Why would BI tools not detect these issues automatically?

BI tools such as **Power BI**, **Tableau**, or **Looker Studio** mainly **visualize and analyze the data they receive**. They do not automatically correct data quality issues.

Reasons include:

* They **assume the source data is already clean and validated**.
* They cannot determine whether a duplicate record is intentional or erroneous.
* They do not know the business meaning of a **NULL** value.
* Text in a numeric field may simply be treated as an error or ignored rather than corrected.
* Mixed date formats may be interpreted incorrectly or imported as text, leading to incorrect filtering and reporting.

### Conclusion

Loading unclean data into a BI dashboard can result in:

* **Incorrect Total Sales KPIs**
* **Inflated or understated revenue**
* **Misleading trends and reports**
* **Poor business decisions based on inaccurate data**

Therefore, data should always be **validated and cleaned during the ETL process before it is loaded into the database or connected to BI tools**.
